# MongoDB Atlas, Basic MQL, and Document Modeling

[![Open Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Download this notebook, open Colab, and choose **File > Upload notebook**. The full draft course is distributed separately from the public Week 1 repository.


This notebook introduces MongoDB through direct, visible operations. You will
insert a small synthetic fixture, query nested fields and arrays, update one test
document, interpret write results, and compare embedded and referenced shapes.

The default path uses `mongomock`, an in-memory teaching substitute. It supports
the operations used here but does not reproduce Atlas networking, indexes'
performance, durability, or every MongoDB feature. The same query cells also run
against Atlas. You do not need to complete both paths.

Run the cells in order. The supplied data cell is setup, not code you need to
memorize. After the worked query, you will change one filter and projection.

## Connection Setup

For local mode, leave `USE_ATLAS = False` and run the setup cells. No account or
network rule is needed. Installing the packages still requires internet access.

For Atlas, use your own **Free** cluster. Set `USE_ATLAS = True` below and run
the first setup cell to obtain this runtime's public IPv4 address and a practice
database name. Colab runs Python on Google's computer, so **Add Current IP** in
your laptop's browser may add the wrong address. The optional IP check contacts
the public ipify service; it sends no database credentials.

Do not set `tlsInsecure=True`. A TLS error is a signal to check the current driver,
URI, DNS, network rule, system time, and certificate path.

In [ ]:
%pip -q install "pymongo>=4.13,<5" "mongomock>=4.3,<5"

In [ ]:
from datetime import datetime, timezone
from getpass import getpass
from ipaddress import IPv4Address
from urllib.request import urlopen
from uuid import uuid4

import mongomock
from pymongo import MongoClient
from pymongo.server_api import ServerApi

USE_ATLAS = False  # Choose one path; the query cells are the same.
DATABASE_NAME = "cst4714_mql_" + uuid4().hex[:8]
print("Practice database:", DATABASE_NAME)

if USE_ATLAS:
    try:
        with urlopen("https://api.ipify.org", timeout=10) as response:
            runtime_ip = str(IPv4Address(response.read().decode().strip()))
        print("Temporary Atlas IP access-list entry:", runtime_ip + "/32")
    except Exception:
        raise RuntimeError(
            "The runtime IP check failed. Retry this cell or use local mode; "
            "do not replace the rule with access from everywhere."
        ) from None

**Atlas only: pause here before the connection cell.** In your Atlas project,
open **Network Access / IP Access List**, add the printed address as a temporary
entry, and wait for it to become active. A `/32` rule allows one IPv4 address.
If the runtime restarts or its outgoing route changes, check the address again.

In **Database Access**, use a database user with read/write access to the printed
practice database. This is different from your Atlas website login. In the
cluster's **Connect > Drivers** instructions, select Python and copy the URI.
Replace the password placeholder with that database user's password. Reserved
password characters inside a URI need percent encoding; do not paste a password
into a code cell. Enter the finished URI only in the hidden prompt below.

`MongoClient` creates a connection manager. `ping` sends an actual command and
checks that the deployment responds. Successful ping does not prove permission
to insert or delete documents; the following cells test those operations.

Use a fresh practice name only after cleaning up the previous run. The database
and collection variables below are handles; the first write creates stored data.

In [ ]:
client = None
mongodb_uri = None
if USE_ATLAS:
    try:
        mongodb_uri = getpass("Paste your Atlas driver URI (hidden): ")
        client = MongoClient(
            mongodb_uri,
            tls=True,
            tlsInsecure=False,
            server_api=ServerApi("1", strict=True, deprecation_errors=True),
            serverSelectionTimeoutMS=10000,
            timeoutMS=10000,
        )
        client.admin.command("ping")
        print("MongoDB responded to ping.")
    except Exception:
        if client is not None:
            client.close()
        raise RuntimeError(
            "Atlas connection failed. Check deployment readiness, database "
            "user/password, the runtime IP rule, driver URI, and DNS/TLS. "
            "Keep certificate verification enabled. Local mode remains available."
        ) from None
    finally:
        mongodb_uri = None  # Do not retain a second copy of the URI in this variable.
else:
    client = mongomock.MongoClient()
    print("Using the offline in-memory MongoDB-compatible path.")

database = client[DATABASE_NAME]
tickets = database["tickets"]

## 1. Load a Small, Reproducible Fixture

Every course document carries `course_fixture: "cst4714"`. Rerunning this data cell
replaces only those marked records in this run's private `tickets` collection.
The last cell removes that entire practice collection and closes the client.
Do not put personal or project data in this temporary collection.

The six tickets use BSON dates through Python `datetime` values, nested requester
documents, tag arrays, and embedded event arrays. This is a small teaching
adaptation of the CSV case: `event_type` becomes `type`, `event_at` becomes `at`,
and the supplied `actor_role` describes the actor. It is not a lossless import
of every CSV column. Empty arrays on 1005 and 1006 mean this fixture omits their
history, not that the original tickets never had events.

In [ ]:
tickets.delete_many({"course_fixture": "cst4714"})
tickets.create_index("ticket_id", unique=True)

fixture = [
    {
        "ticket_id": 1001,
        "category": "streetlight",
        "priority": "high",
        "status": "open",
        "subject": "Streetlight dark near bus stop",
        "requester": {"user_id": 101, "display_name": "Maya Chen"},
        "assignee_id": 201,
        "opened_at": datetime(2026, 2, 1, 23, 10, tzinfo=timezone.utc),
        "tags": ["lighting", "safety"],
        "events": [
            {"event_id": 5001, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 1, 23, 10, tzinfo=timezone.utc)},
            {"event_id": 5002, "type": "assigned", "actor_role": "agent",
             "at": datetime(2026, 2, 2, 14, 5, tzinfo=timezone.utc)},
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1002,
        "category": "sanitation",
        "priority": "medium",
        "status": "in_progress",
        "subject": "Missed recycling pickup",
        "requester": {"user_id": 102, "display_name": "Luis Rivera"},
        "assignee_id": 202,
        "opened_at": datetime(2026, 2, 2, 15, 45, tzinfo=timezone.utc),
        "tags": ["recycling"],
        "events": [
            {"event_id": 5003, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 2, 15, 45, tzinfo=timezone.utc)},
            {"event_id": 5004, "type": "status_changed", "actor_role": "agent",
             "at": datetime(2026, 2, 3, 13, 30, tzinfo=timezone.utc)},
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1003,
        "category": "water",
        "priority": "urgent",
        "status": "resolved",
        "subject": "Low water pressure",
        "requester": {"user_id": 103, "display_name": "Amina Yusuf"},
        "assignee_id": 201,
        "opened_at": datetime(2026, 2, 3, 12, 5, tzinfo=timezone.utc),
        "tags": ["water", "building"],
        "events": [
            {"event_id": 5005, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 3, 12, 5, tzinfo=timezone.utc)},
            {"event_id": 5006, "type": "status_changed", "actor_role": "agent",
             "at": datetime(2026, 2, 3, 14, 25, tzinfo=timezone.utc)},
            {"event_id": 5007, "type": "status_changed", "actor_role": "agent",
             "at": datetime(2026, 2, 3, 19, 40, tzinfo=timezone.utc)},
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1004,
        "category": "parks",
        "priority": "low",
        "status": "new",
        "subject": "Broken bench slat",
        "requester": {"user_id": 104, "display_name": "Jordan Bell"},
        "assignee_id": None,
        "opened_at": datetime(2026, 2, 4, 17, 20, tzinfo=timezone.utc),
        "tags": ["parks"],
        "events": [
            {"event_id": 5008, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 4, 17, 20, tzinfo=timezone.utc)}
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1005,
        "category": "sanitation",
        "priority": "high",
        "status": "resolved",
        "subject": "Overflowing corner bin",
        "requester": {"user_id": 101, "display_name": "Maya Chen"},
        "assignee_id": 202,
        "opened_at": datetime(2026, 2, 5, 14, 0, tzinfo=timezone.utc),
        "tags": ["sanitation", "safety"],
        "events": [],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1006,
        "category": "streetlight",
        "priority": "medium",
        "status": "in_progress",
        "subject": "Flickering lamp outside library",
        "requester": {"user_id": 102, "display_name": "Luis Rivera"},
        "assignee_id": 201,
        "opened_at": datetime(2026, 2, 6, 1, 30, tzinfo=timezone.utc),
        "tags": ["lighting", "library"],
        "events": [],
        "course_fixture": "cst4714",
    },
]

insert_result = tickets.insert_many(fixture)
print("Inserted documents:", len(insert_result.inserted_ids))
print("Verified fixture count:", tickets.count_documents({"course_fixture": "cst4714"}))

## 2. Filter, Project, and Sort

The result grain is one document per matching ticket. The first dictionary is
the filter: every listed field condition must hold. `$in` supplies alternatives
for one field. The second dictionary is the projection: `1` includes a field,
while `_id: 0` suppresses the otherwise included identifier. These choices change
the returned view, not the stored document.

`find` returns a cursor that the `for` loop reads. This example orders by opening
time descending. The six-ticket fixture has one active high/urgent ticket: 1001.
Ticket 1003 is urgent but resolved, so it does not pass the status filter.

In [ ]:
active_high_priority = tickets.find(
    {
        "course_fixture": "cst4714",
        "status": {"$in": ["new", "open", "in_progress"]},
        "priority": {"$in": ["high", "urgent"]},
    },
    {"_id": 0, "ticket_id": 1, "priority": 1, "status": 1, "subject": 1, "opened_at": 1},
).sort("opened_at", -1)

for document in active_high_priority:
    print(document)

### Your Turn

Modify the next filter to choose a different status set or category, and modify the
projection to add exactly one useful field. State the expected result grain before
running it.

In [ ]:
# Grain: one document per matching ticket.
for document in tickets.find(
    {"course_fixture": "cst4714", "category": "streetlight"},
    {"_id": 0, "ticket_id": 1, "category": 1, "status": 1, "subject": 1},
).sort("ticket_id", 1):
    print(document)

## 3. Query a Nested Field and an Array

Dot notation reaches `requester.user_id`. Equality against an array field matches
when the array contains that value.

In [ ]:
print("Tickets requested by user 101:")
for document in tickets.find(
    {"course_fixture": "cst4714", "requester.user_id": 101},
    {"_id": 0, "ticket_id": 1, "requester.display_name": 1, "status": 1},
):
    print(document)

print("\nTickets tagged safety:")
for document in tickets.find(
    {"course_fixture": "cst4714", "tags": "safety"},
    {"_id": 0, "ticket_id": 1, "tags": 1},
):
    print(document)

## 4. `$elemMatch` Requires Conditions on the Same Array Element

The question asks for one event whose type is `status_changed` **and** whose actor
role is `agent`. `$elemMatch` prevents one array element from satisfying the type
while a different element satisfies the actor condition.

First inspect a counterexample. Ticket 1001 has a resident-created event and a
different agent-assigned event. Separate dotted predicates for `created` and
`agent` can match those different events. A request for an event *created by an
agent* must require both conditions on the same element.

In [ ]:
separate_elements = {
    "course_fixture": "cst4714",
    "events.type": "created",
    "events.actor_role": "agent",
}
same_element = {
    "course_fixture": "cst4714",
    "events": {"$elemMatch": {"type": "created", "actor_role": "agent"}},
}
print("Separate conditions:", [d["ticket_id"] for d in tickets.find(separate_elements)])
print("Same event required:", [d["ticket_id"] for d in tickets.find(same_element)])
# Expect [1001, 1002, 1003] versus []. No fixture event was created by an agent.

print("Status changes made by an agent:")
for document in tickets.find(
    {
        "course_fixture": "cst4714",
        "events": {
            "$elemMatch": {"type": "status_changed", "actor_role": "agent"}
        },
    },
    {"_id": 0, "ticket_id": 1, "events": 1},
):
    print(document)

## 5. Verify Matched and Modified Counts

This cell resets only test ticket 1099, then inserts it in state `new`. Rerunning
the setup cell starts this small write experiment again without making duplicates.
The unique index prevents two documents from sharing one ticket number.

In [ ]:
test_filter = {"ticket_id": 1099, "test_record": True, "course_fixture": "cst4714"}
tickets.delete_many(test_filter)
test_document = {
    "ticket_id": 1099,
    "category": "parks",
    "priority": "low",
    "status": "new",
    "subject": "Disposable MQL test",
    "requester": {"user_id": 101, "display_name": "Maya Chen"},
    "opened_at": datetime.now(timezone.utc),
    "events": [],
    "test_record": True,
    "course_fixture": "cst4714",
}
tickets.insert_one(test_document)

Run the next cell **twice without rerunning the insertion cell**. `$set` changes
only its named fields. The first execution should report `1 1`: one match and one
modified document. The second should report `1 0`: the same document already has
the requested values. The read-back checks what is actually stored.

In [ ]:
first_update = tickets.update_one(
    test_filter,
    {"$set": {"status": "in_progress", "assignee_id": 202}},
)
print("Matched/modified:", first_update.matched_count, first_update.modified_count)
print(tickets.find_one(test_filter, {"_id": 0, "ticket_id": 1, "status": 1, "assignee_id": 1}))

Now compare an **expected-state filter**. This operation asks to change a ticket
only while it is `new`. Our ticket is already `in_progress`, so `0 0` is correct.
An empty match is different from a match whose values did not change.

In [ ]:
stale_update = tickets.update_one(
    {**test_filter, "status": "new"},
    {"$set": {"status": "resolved"}},
)
print("Expected-state filter matched/modified:", stale_update.matched_count, stale_update.modified_count)
print("Status is still:", tickets.find_one(test_filter)["status"])

## 6. Append One Event and Read Back the Final Document

`$push` appends to the array. This example also requires event 5999 to be absent.
Rerunning the cell therefore does not append the same event again. That narrow
guard works for this one document; it is not a complete event-processing system.

This append and the earlier status change are **two separate writes**. A reader
between them could see the new status without the event. Day 2 shows how one
update can change status and append an embedded event atomically.

In [ ]:
event_update = tickets.update_one(
    {**test_filter, "events.event_id": {"$ne": 5999}},
    {
        "$push": {
            "events": {
                "event_id": 5999,
                "type": "status_changed",
                "actor_role": "agent",
                "at": datetime.now(timezone.utc),
            }
        }
    },
)
print("Event append matched/modified:", event_update.matched_count, event_update.modified_count)
print(tickets.find_one(test_filter, {"_id": 0}))

## 7. Delete Only the Disposable Record

The predicate includes both the identifier and the safety marker. The final query
confirms cleanup.

In [ ]:
print("Preview:", tickets.find_one(test_filter, {"_id": 0, "ticket_id": 1}))
delete_result = tickets.delete_one(test_filter)
print("Deleted count:", delete_result.deleted_count)
print("Remaining test record:", tickets.find_one(test_filter))
print("Other fixture tickets:", tickets.count_documents({"course_fixture": "cst4714"}))

## 8. Compare Two Models

These are partial design sketches, not additional database writes. The referenced
event stores `ticket_id`; the parent does not need a list that grows by one ID
for every event. Day 2 develops the read and update consequences.

In [ ]:
embedded_ticket = {
    "ticket_id": 1001,
    "requester": {"user_id": 101, "display_name": "Maya Chen"},
    "events": [{"event_id": 5001, "type": "created"}],
}

referenced_ticket = {
    "ticket_id": 1001,
    "requester_id": 101,
}
referenced_event = {"event_id": 5001, "ticket_id": 1001, "type": "created"}

print("Embedded sketch:", embedded_ticket)
print("Referenced sketch:", referenced_ticket)
print("Separate event sketch:", referenced_event)

## Results Record: Complete Before Submission

In a short explanation, name your changed filter/projection and one returned
ticket, explain the array counterexample using actual IDs, and interpret the
first and repeated write counts, compared with the expected-state filter. Explain why the delete targeted only the test
record. If you used local mode, say so; no Atlas account task is required on that
path. Day 2's assignment develops the model comparison further.

Before submitting, remove any accidentally saved credential from source or output.

**License:** prose CC BY-NC-SA 4.0; code MIT; synthetic data CC0.

In [ ]:
try:
    database.drop_collection("tickets")
    print("Removed the tickets collection from this run's practice database.")
finally:
    client.close()
    mongodb_uri = None
print("The database client is closed. Later lessons supply fresh fixtures.")
if USE_ATLAS:
    print("In Atlas, remove the temporary IP rule and any class-only database user.")
    print("This cleanup does not delete or stop your Free cluster.")